# HARP Experiment Analysis

This notebook provides analysis and visualization for the HARP traffic matrix experiments.

In [ ]:
import csv
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Set plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Load Experiment Results

In [ ]:
def load_results(csv_path):
    """Load results from mlu_vs_loss_results.csv"""
    results = []
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row['success'] == 'True':
                results.append({
                    'tm_id': int(row['tm_id']),
                    'gurobi_mlu': float(row['gurobi_mlu']),
                    'avg_loss': float(row['avg_loss_rate']) * 100,
                    'max_loss': float(row['max_loss_rate']) * 100,
                    'bytes_sent': int(row['total_bytes_sent']),
                    'bytes_dropped': int(row['total_bytes_dropped']),
                })
    return results

# Example: Load overlapping experiment results
# results = load_results('exp4_optimal_55s/mlu_vs_loss_results.csv')

## Plot: Loss Rate per TM (Overlapping Experiment)

In [ ]:
def plot_loss_per_tm(results, title='Loss Rate per Traffic Matrix'):
    """Bar chart of loss rate per TM."""
    tm_ids = [r['tm_id'] for r in results]
    avg_loss = [r['avg_loss'] for r in results]
    
    colors = ['#2ecc71' if l < 5 else '#f39c12' if l < 20 else '#e74c3c' for l in avg_loss]
    
    fig, ax = plt.subplots()
    ax.bar(range(len(tm_ids)), avg_loss, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Traffic Matrix ID', fontsize=14)
    ax.set_ylabel('Average Loss Rate (%)', fontsize=14)
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xticks(range(len(tm_ids)))
    ax.set_xticklabels(tm_ids)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    return fig

# Example:
# results = load_results('exp4_optimal_55s/mlu_vs_loss_results.csv')
# plot_loss_per_tm(results)

## Plot: Loss vs Time (Multiple Simulations)

In [ ]:
def plot_loss_vs_time(time_dirs):
    """Plot total loss rate vs simulation time.
    
    Args:
        time_dirs: dict of {time_seconds: 'exp4_time_Xs/'}
    """
    times = []
    loss_rates = []
    
    for t, dir_name in sorted(time_dirs.items()):
        csv_path = Path(dir_name) / 'mlu_vs_loss_results.csv'
        if csv_path.exists():
            results = load_results(csv_path)
            total_sent = sum(r['bytes_sent'] for r in results)
            total_dropped = sum(r['bytes_dropped'] for r in results)
            loss_rate = total_dropped / total_sent * 100 if total_sent > 0 else 0
            times.append(t)
            loss_rates.append(loss_rate)
    
    fig, ax = plt.subplots()
    ax.plot(times, loss_rates, 'o-', color='#e74c3c', linewidth=3, markersize=10)
    ax.fill_between(times, loss_rates, alpha=0.2, color='#e74c3c')
    ax.set_xlabel('Simulation Time (seconds)', fontsize=14)
    ax.set_ylabel('Total Loss Rate (%)', fontsize=14)
    ax.set_title('Network Loss Increases with Overlapping TMs', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

# Example:
# time_dirs = {10: 'exp4_time_10s', 20: 'exp4_time_20s', 30: 'exp4_time_30s'}
# plot_loss_vs_time(time_dirs)

## Plot: Default vs Optimal Routing Comparison

In [ ]:
def plot_routing_comparison(default_csv, optimal_csv):
    """Compare default vs optimal routing side by side."""
    default = load_results(default_csv)
    optimal = load_results(optimal_csv)
    
    tm_ids = [r['tm_id'] for r in default]
    default_loss = [r['avg_loss'] for r in default]
    optimal_loss = [r['avg_loss'] for r in optimal]
    
    x = np.arange(len(tm_ids))
    width = 0.35
    
    fig, ax = plt.subplots()
    ax.bar(x - width/2, default_loss, width, label='Default Routing', color='#e74c3c', alpha=0.8)
    ax.bar(x + width/2, optimal_loss, width, label='Optimal (Gurobi)', color='#3498db', alpha=0.8)
    
    ax.set_xlabel('Traffic Matrix ID', fontsize=14)
    ax.set_ylabel('Average Loss Rate (%)', fontsize=14)
    ax.set_title('Default vs Optimal Routing (Overlapping TMs)', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(tm_ids)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    return fig

# Example:
# plot_routing_comparison('exp4_default_10/mlu_vs_loss_results.csv', 
#                         'exp4_optimal_10/mlu_vs_loss_results.csv')

## Summary Statistics

In [ ]:
def print_summary(csv_path):
    """Print summary statistics for an experiment."""
    results = load_results(csv_path)
    
    total_sent = sum(r['bytes_sent'] for r in results)
    total_dropped = sum(r['bytes_dropped'] for r in results)
    avg_loss = np.mean([r['avg_loss'] for r in results])
    max_loss = max([r['max_loss'] for r in results])
    
    print(f"Experiment: {csv_path}")
    print(f"  TMs: {len(results)}")
    print(f"  Total sent: {total_sent/1e9:.2f} GB")
    print(f"  Total dropped: {total_dropped/1e6:.1f} MB")
    print(f"  Avg loss rate: {avg_loss:.2f}%")
    print(f"  Max loss rate: {max_loss:.2f}%")

# Example:
# print_summary('exp4_optimal_55s/mlu_vs_loss_results.csv')